# Notebook 10 ? Cocokkan Nama yang Tidak Persis Sama

Bantu mencari pasangan nama yang mirip, misalnya nama pelanggan yang ejaannya sedikit berbeda. Hasilnya adalah daftar kandidat untuk ditinjau manusia; kecocokan tidak diterapkan otomatis karena skor kemiripan bisa salah.

**Pustaka tambahan:** jalankan `pip install rapidfuzz` pada lingkungan Python notebook jika belum tersedia. Fuzzy matching dapat memakan waktu pada file besar; mulai dari sampel kecil.

In [ ]:
from pathlib import Path
import pandas as pd
try:
    from rapidfuzz import process, fuzz
except ImportError as exc:
    raise ImportError("Pustaka rapidfuzz belum tersedia. Pasang dengan: pip install rapidfuzz") from exc

# KONFIGURASI
LEFT_FILE = Path("Input/sumber_a.csv")
RIGHT_FILE = Path("Input/sumber_b.csv")
LEFT_NAME = "nama"
RIGHT_NAME = "nama"
LEFT_ID = "id"       # Bisa None jika tidak ada kolom ID
RIGHT_ID = "id"      # Bisa None jika tidak ada kolom ID
MIN_SCORE = 85        # Skor 0-100; naikkan jika ingin hasil lebih ketat
MIN_GAP = 5           # Selisih minimal terhadap kandidat kedua; turunkan untuk melihat kandidat ambigu
LIMIT_ROWS = None     # Contoh 500 untuk uji sampel; None untuk semua baris
OUTPUT_PATH = Path("Output/hasil_kandidat_pencocokan.csv")
def find_project_dir():
    current = Path.cwd().resolve()
    for folder in (current, *current.parents):
        if (folder / "Input").is_dir():
            (folder / "Output").mkdir(parents=True, exist_ok=True)
            return folder
    return current

BASE_DIR = find_project_dir()


## Cari kandidat terbaik untuk ditinjau

In [ ]:
def read_input(path):
    path = path if path.is_absolute() else BASE_DIR / path
    if not path.exists(): raise FileNotFoundError(f"File tidak ditemukan: {path}")
    return pd.read_csv(path, dtype="string", encoding="utf-8-sig")
left, right = read_input(LEFT_FILE), read_input(RIGHT_FILE)
for frame, col in [(left, LEFT_NAME), (right, RIGHT_NAME)]:
    if col not in frame: raise KeyError(f"Kolom nama tidak ditemukan: {col}")
if LEFT_ID and LEFT_ID not in left: raise KeyError(f"Kolom ID kiri tidak ditemukan: {LEFT_ID}")
if RIGHT_ID and RIGHT_ID not in right: raise KeyError(f"Kolom ID kanan tidak ditemukan: {RIGHT_ID}")
if LIMIT_ROWS is not None: left = left.head(LIMIT_ROWS)
right = right.copy()
right["_match_text"] = right[RIGHT_NAME].fillna("").astype(str).str.casefold().str.strip()
choices = right["_match_text"].tolist()
results = []
for _, row in left.iterrows():
    original = "" if pd.isna(row[LEFT_NAME]) else str(row[LEFT_NAME]).strip()
    query = original.casefold()
    hits = process.extract(query, choices, scorer=fuzz.WRatio, limit=2) if query else []
    best = hits[0] if hits else None
    second_score = hits[1][1] if len(hits) > 1 else -1
    idx = best[2] if best else None
    score = best[1] if best else 0
    candidate = right.iloc[idx] if idx is not None else None
    accepted_candidate = bool(best and score >= MIN_SCORE and score - second_score >= MIN_GAP)
    item = {"nama_kiri": original, "nama_kandidat": candidate[RIGHT_NAME] if candidate is not None else pd.NA,
            "skor": score, "skor_kandidat_kedua": second_score,
            "saran": "periksa kandidat" if accepted_candidate else "perlu tinjauan manual"}
    if LEFT_ID: item["id_kiri"] = row[LEFT_ID]
    if RIGHT_ID: item["id_kandidat"] = candidate[RIGHT_ID] if candidate is not None else pd.NA
    results.append(item)
result = pd.DataFrame(results)
display(result.head(30))
print(f"Kandidat dibuat: {len(result):,}. Semua hasil perlu diperiksa sebelum dipakai untuk mengubah data.")


In [ ]:
output = OUTPUT_PATH if OUTPUT_PATH.is_absolute() else BASE_DIR / OUTPUT_PATH
output.parent.mkdir(parents=True, exist_ok=True)
result.to_csv(output, index=False, encoding="utf-8-sig")
print(f"Tersimpan: {output.resolve()}")